# M4 — MoE Top-2
Gắn M1 token artifacts làm Input. Notebook huấn luyện Dense baseline và MoE từ đầu trên cùng split/config/budget để so sánh công bằng. Sau khi push code, điền SHA đầy đủ bên dưới rồi chạy tuần tự.

In [ ]:
REPO_URL = 'https://github.com/shinkbb/mini-gpt.git'
COMMIT_SHA = 'REPLACE_WITH_FULL_M4_COMMIT_SHA'
M1_INPUT = '/kaggle/input'  # recursively find attached M1 tokenizer and token tensors
MAX_STEPS = 1000
DEVICE = 'auto'
PRECISION = 'fp16'

In [ ]:
from pathlib import Path
import re, shutil, subprocess, sys
assert re.fullmatch(r'[0-9a-fA-F]{40}', COMMIT_SHA), 'Set full M4 commit SHA after pushing'
root = Path('/kaggle/working/mini-gpt-m4')
subprocess.run(['git', 'clone', REPO_URL, str(root)], check=True)
subprocess.run(['git', 'fetch', 'origin', COMMIT_SHA], cwd=root, check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT_SHA], cwd=root, check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'], cwd=root, text=True).strip().lower() == COMMIT_SHA.lower()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=root, check=True)
input_root = Path(M1_INPUT)
m1_candidates = [p for p in input_root.rglob('train_tokens.pt') if (p.parent/'validation_tokens.pt').is_file() and (p.parent/'tokenizer.json').is_file()]
assert m1_candidates, 'Attach the M1 artifact dataset (tokenizer.json + train/validation_tokens.pt)'
m1_dir = m1_candidates[0].parent
(root/'artifacts').mkdir(exist_ok=True)
for name in ('tokenizer.json','train_tokens.pt','validation_tokens.pt'): shutil.copy2(m1_dir/name, root/'artifacts'/name)
print('Source:', COMMIT_SHA, 'M1 artifacts:', m1_dir)

In [ ]:
import yaml
moe_config = yaml.safe_load((root/'configs/moe.yaml').read_text())
moe_config['training']['max_steps'] = MAX_STEPS
moe_config['training']['precision'] = PRECISION
(root/'configs/moe.yaml').write_text(yaml.safe_dump(moe_config, sort_keys=False))
dense_config = yaml.safe_load((root/'configs/dense.yaml').read_text())
dense_config['training']['max_steps'] = MAX_STEPS
dense_config['training']['precision'] = PRECISION
dense_config['training']['output_dir'] = 'runs/dense_m4_control'
(root/'configs/dense_m4.yaml').write_text(yaml.safe_dump(dense_config, sort_keys=False))
dense_ckpt = root/'runs/dense_m4_control/best.pt'
if not dense_ckpt.exists():
    subprocess.run([sys.executable, 'train.py', '--config', 'configs/dense_m4.yaml', '--max-steps', str(MAX_STEPS), '--device', DEVICE], cwd=root, check=True)
else:
    print('Reuse completed Dense control checkpoint:', dense_ckpt)
moe_last = root/'runs/moe/last.pt'
moe_args = [sys.executable, 'train.py', '--config', 'configs/moe.yaml', '--max-steps', str(MAX_STEPS), '--device', DEVICE]
if moe_last.exists(): moe_args += ['--resume', str(moe_last)]
subprocess.run(moe_args, cwd=root, check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'experiments.benchmark_moe', '--dense-checkpoint', 'runs/dense_m4_control/best.pt', '--moe-checkpoint', 'runs/moe/best.pt', '--validation-tokens', 'artifacts/validation_tokens.pt', '--precision', PRECISION, '--device', DEVICE, '--batches', '100', '--repeats', '3', '--dense-training-metrics', 'runs/dense_m4_control/metrics.jsonl', '--moe-training-metrics', 'runs/moe/metrics.jsonl', '--output', 'reports/benchmark_moe.json'], cwd=root, check=True)
print('Save Version to preserve Dense/MoE checkpoints and reports/benchmark_moe.json/.csv/.svg as Kaggle Notebook Output.')